# Part 4 Explained: Model Evaluation And Interpretation

This notebook explains, in simple and detailed language, what we did in `04_model_evaluation_interpretation.ipynb`.

Use this as your study notebook for viva, presentation preparation, and understanding the full logic behind Part 4.

The main question in Part 4 is:

> After building clustering and anomaly detection models in Part 3, how do we evaluate them, interpret them, and connect them to real urban mobility decisions?


## 1. What Part 4 Is About

Part 4 is not about creating new models. That was Part 3.

Part 4 is about proving that the models are meaningful.

The exam requirement asks us to:

- evaluate models using suitable metrics;
- visualize results using plots, maps, or charts;
- interpret the numbers;
- discuss what the findings mean for urban mobility;
- explain limitations honestly.

So in Part 4, we evaluate two model families:

| Model family | Models used | Main purpose |
|---|---|---|
| Clustering | KMeans, DBSCAN | Find bus reliability patterns and hotspots |
| Anomaly detection | Isolation Forest, LOF | Find unusual bus reliability windows |


## 2. Project Scope Reminder: Bus-Only HSL Reliability

The raw HSL feed contains several public transport modes, but our project is focused only on buses.

We use these GTFS `route_type` values:

| Route type | Meaning in this project |
|---:|---|
| `701` | HSL bus-family service |
| `702` | HSL bus-family service variant |
| `704` | HSL bus-family service variant |

This bus-only filtering is important because buses behave differently from metro, tram, rail, and ferry services.

For example, bus delays are affected by road traffic, signals, roadworks, weather, boarding time, and congestion. If we mixed all modes together, the model would be harder to explain.

Presentation sentence:

> I kept the project bus-only so the reliability analysis is consistent and explainable. The models evaluate HSL bus route-stop-hour behavior, not the whole multimodal HSL network.


## 3. What Data Part 4 Loads

Part 4 loads the same model outputs used by the dashboard.

| API/data source | What it gives us | Why it is needed |
|---|---|---|
| `/models/status` | model counts, latest buckets, coverage | proves model freshness and data volume |
| `/ingestion/status` | latest GTFS-RT poll, stored vehicle/trip counts | proves live data is being collected |
| `/models/clusters` | KMeans and DBSCAN cluster map points | used for cluster interpretation and maps |
| `/models/anomalies/live` | anomaly-scored route-stop-hour windows | used for anomaly interpretation |
| `/models/runs` | tuning results and selected model parameters | used for evaluation metrics |
| `/routes` | route metadata | used to validate bus-only route types |

This is important because the notebook and dashboard are connected to the same backend. The notebook is not a separate fake analysis; it evaluates the same outputs shown on the live dashboard.


## 4. What Is A Model Feature Window?

The raw GTFS and GTFS-RT data is transformed into model-ready rows called **route-stop-hour feature windows**.

A feature window means something like:

> Route 510 at Stop X between 08:00 and 09:00 had average delay 1.8 minutes, prediction coverage 96%, 22 observations, and stable feed freshness.

Each row contains features such as:

- mean delay;
- median delay;
- 95th percentile absolute delay;
- delay standard deviation;
- on-time ratio;
- late ratio;
- early ratio;
- prediction coverage ratio;
- vehicle observation count;
- feed freshness;
- hour, weekday, weekend flag;
- stop latitude and longitude.

Why this matters:

> Machine learning models cannot directly understand raw GTFS files. They need structured numerical features. Feature windows are the bridge between raw transit data and data mining.


## 5. Bus-Only Validation In Part 4

Before evaluating models, we verify that the loaded data contains only bus route types.

This validation checks route types in:

- route API output;
- cluster layer output;
- anomaly layer output.

If any non-bus route type appears, the notebook raises an error.

Why this is important:

> It proves that our evaluation is consistent with our project scope. We are not accidentally evaluating tram, metro, train, or ferry data.


## 6. Why Clustering Evaluation Is Different From Classification Evaluation

In supervised classification, we usually have true labels. For example, we may know whether each bus was delayed or not.

But clustering is unsupervised. There is no official answer saying:

- this stop belongs to cluster 1;
- this stop belongs to cluster 2;
- this point is definitely a hotspot.

Therefore, clustering is evaluated using internal metrics and interpretability.

We ask:

- Are the clusters separated well?
- Are clusters compact?
- Do the clusters make operational sense?
- Can we explain them as reliable, watch, delay hotspot, or low realtime coverage?


## 7. KMeans Evaluation

KMeans groups bus route-stop-hour windows into a fixed number of clusters, called `k`.

In Part 4, we evaluate several values of `k`, usually from 2 to 10.

The notebook reports three metrics:

| Metric | Meaning | Better direction |
|---|---|---|
| Silhouette Score | Are points close to their own cluster and far from other clusters? | Higher is better |
| Davies-Bouldin Index | Are clusters compact and separate, or overlapping? | Lower is better |
| Calinski-Harabasz Score | Is between-cluster separation strong compared with within-cluster variation? | Higher is better |

How to explain KMeans result:

> We tested different cluster counts and selected a setting that balanced metric performance and interpretability. The goal was not only mathematical quality, but also whether the resulting groups make sense for bus reliability analysis.

Important point:

> A model with the best single metric is not always the best operational model. For urban mobility, interpretability matters because planners need to understand the result.


## 8. DBSCAN Evaluation

DBSCAN is used to find dense hotspot patterns.

Unlike KMeans, DBSCAN does not force every point into a cluster. It can mark weak or isolated points as noise.

DBSCAN uses two important parameters:

| Parameter | Meaning |
|---|---|
| `eps` | How close points must be to count as neighbors |
| `min_samples` | Minimum nearby points needed to form a dense cluster |

Part 4 evaluates DBSCAN using:

- cluster count;
- noise share;
- Silhouette Score;
- Davies-Bouldin Index;
- Calinski-Harabasz Score.

How to explain DBSCAN:

> DBSCAN is useful for public transport because reliability problems often appear in spatially concentrated areas, such as corridors, terminals, and busy stops. It helps identify where planners should investigate first.

In our dashboard and notebook, DBSCAN hotspots are shown as stronger hotspot points.


## 9. Cluster Labels: What They Mean

The project converts model output into human-readable reliability labels.

| Label | Meaning | Operational interpretation |
|---|---|---|
| `reliable` | Good realtime coverage and stable delay behavior | Healthy-looking bus reliability window |
| `watch` | Some weaker signals but not a severe problem | Monitor this route/stop/time further |
| `delay_hotspot` | Higher or concentrated delay behavior | Candidate location for reliability investigation |
| `low_realtime_coverage` | Weak GTFS-RT prediction coverage | Data quality or feed completeness issue |

Important distinction:

- `delay_hotspot` suggests possible service reliability problem.
- `low_realtime_coverage` suggests possible realtime data quality problem.

Presentation sentence:

> The labels translate mathematical clusters into transport planning language, so the result is easier to explain to non-technical stakeholders.


## 10. Why Anomaly Detection Uses Pseudo-Labels

For anomaly detection, we do not have official ground-truth labels from HSL saying:

- this window is definitely an anomaly;
- this window is definitely normal.

So Part 4 uses pseudo-labels.

Pseudo-labels are rule-based indicators of suspicious operation.

Examples:

- extreme delay window;
- late-heavy window;
- prediction coverage dropout;
- stale feed window.

This is acceptable for an academic MVP if we explain the limitation clearly.

Presentation sentence:

> Since official anomaly labels were not available, I used transparent pseudo-labels based on operational warning signs. These are not perfect ground truth, but they allow practical evaluation of the anomaly models.


## 11. Isolation Forest Evaluation

Isolation Forest detects globally unusual points.

The idea is:

> Unusual points are easier to isolate from the rest of the data.

In this project, Isolation Forest looks for route-stop-hour windows that are unusual based on delay, coverage, freshness, and related features.

Part 4 evaluates Isolation Forest using:

| Metric | Meaning |
|---|---|
| Precision@K | Among top alerts, how many match pseudo-label warning signs |
| Recall | How much pseudo-labelled suspicious behavior was captured |
| F1 proxy | Balance between precision and recall |
| False-alarm share | Share of flags that may be unnecessary |

How to explain:

> Isolation Forest is good for detecting globally unusual bus reliability windows. It is useful as a watchlist model because it can surface unusual delay or data quality behavior quickly.


## 12. LOF Evaluation

LOF means Local Outlier Factor.

LOF detects points that are unusual compared with their local neighborhood.

This is useful because bus behavior is local:

- a 5-minute delay may be normal on one route during rush hour;
- the same 5-minute delay may be unusual on another route or time period.

Part 4 evaluates LOF using the same anomaly metrics:

- Precision@K;
- recall;
- F1 proxy;
- false-alarm share.

How to explain:

> LOF complements Isolation Forest by focusing on local abnormality. It helps detect bus windows that look unusual compared with similar nearby or related observations.


## 13. Why Precision, Recall, And False Alarms Matter

In anomaly detection, there is always a trade-off.

If the model flags too many windows, planners may ignore the alerts.

If it flags too few windows, important problems may be missed.

That is why Part 4 reports several metrics together.

| Metric | Why it matters for transit operations |
|---|---|
| Precision@K | Top alerts should be meaningful because humans may inspect them first |
| Recall | The model should capture enough suspicious behavior |
| F1 proxy | Gives a balance between precision and recall |
| False-alarm share | Too many false alarms reduce trust in the dashboard |

Presentation sentence:

> In urban mobility monitoring, a model is not useful only because it finds many anomalies. It must find anomalies that are credible enough for planners to investigate.


## 14. What The Spatial Visualizations Show

Part 4 includes spatial plots because transit reliability is geographic.

The notebook includes:

- reliability cluster plot by stop location;
- DBSCAN hotspot plot;
- anomaly watchlist plot;
- optional CartoDB basemap plots using `contextily`;
- optional interactive CartoDB map using `folium`.

Why this matters:

> A table can tell us which route has problems, but a map can show where those problems are concentrated.

Presentation sentence:

> Spatial visualization helps connect model outputs to real urban geography, such as corridors, stops, terminals, and service areas.


## 15. Key Insights From Part 4

The most important insights are:

1. Bus reliability can be summarized into interpretable route-stop-hour windows.
2. KMeans gives broad reliability groups.
3. DBSCAN identifies hotspot-style concentrations.
4. Isolation Forest creates a strong global anomaly watchlist.
5. LOF adds local abnormality detection.
6. Prediction coverage is important because missing realtime data affects reliability confidence.
7. The dashboard makes the notebook results visible and explainable.

Do not overclaim.

Correct wording:

> The model identifies reliability patterns and anomaly watchlist windows.

Avoid saying:

> The model proves exact causes of delays.

The project detects patterns; it does not prove causality.


## 16. Urban Mobility Recommendations

Based on Part 4, the recommendations are:

| Finding | Recommendation |
|---|---|
| Delay hotspots | Review timetable padding, road priority, bus lanes, and stop operations |
| Low realtime coverage | Improve GTFS-RT feed reliability and prediction completeness |
| Repeated anomaly windows | Use as a watchlist for planners or operations staff |
| Route-level reliability variation | Compare routes by OTP, delay, and coverage before service planning decisions |
| Limited history | Keep the poller running for more days/weeks before making stronger long-term conclusions |

Presentation sentence:

> The project can help municipalities prioritize where to investigate reliability problems, where realtime data quality needs improvement, and which routes deserve closer operational monitoring.


## 17. Limitations To Explain Honestly

The limitations are not weaknesses if explained clearly.

| Limitation | Meaning |
|---|---|
| Limited GTFS-RT history | Data only accumulates while Docker/poller is running |
| Missing stable trip IDs | Some realtime messages are hard to match perfectly to static schedules |
| Pseudo-labels | Anomaly labels are rule-based, not official ground truth |
| Bus-only scope | Results should not be generalized to metro, tram, ferry, or rail |
| Not predictive yet | The project does pattern discovery and anomaly detection, not future delay prediction |

Good viva answer:

> I treated these as limitations of the MVP. The current project is strong for exploratory reliability analysis, and with longer history it can be extended to predictive modelling.


## 18. How To Present Part 4 In 90 Seconds

Use this script:

> In Part 4, I evaluated the models from Part 3. For clustering, because there are no true labels, I used internal metrics: Silhouette Score, Davies-Bouldin Index, and Calinski-Harabasz Score. I also checked whether the clusters are interpretable as reliable, watch, delay hotspot, or low realtime coverage. For anomaly detection, there were no official HSL anomaly labels, so I used pseudo-labels based on operational warning signs like extreme delay, stale feed, late-heavy windows, and coverage dropouts. I evaluated Isolation Forest and LOF using Precision@K, recall, F1 proxy, and false-alarm share. Finally, I visualized the results on static plots and map layers to show where reliability patterns and anomaly watchlist points appear geographically. The main insight is that the system can support bus reliability monitoring and planning, but longer GTFS-RT history would improve confidence.


## 19. Viva Questions And Answers

**Q: Why did you not use accuracy for clustering?**

Accuracy needs true labels. Clustering is unsupervised, so we use internal metrics like Silhouette, Davies-Bouldin, and Calinski-Harabasz.

**Q: Why use pseudo-labels for anomaly detection?**

Official anomaly labels were not available, so we used transparent operational rules to approximate suspicious windows.

**Q: What does low realtime coverage mean?**

It means the GTFS-RT prediction signal is weak or incomplete. It may be a data quality issue, not necessarily a bus service delay.

**Q: What is the difference between DBSCAN and KMeans?**

KMeans forces all points into a fixed number of clusters. DBSCAN finds dense groups and can leave weak points as noise, making it better for hotspot detection.

**Q: What is the biggest limitation?**

The main limitation is limited historical GTFS-RT data and absence of official anomaly labels.

**Q: What is the main achievement?**

We built a full bus reliability analysis pipeline from ingestion to feature engineering, modelling, evaluation, and dashboard visualization.


## 20. Final Part 4 Conclusion

Part 4 proves that the selected models are evaluated appropriately and interpreted in a practical urban mobility context.

The final conclusion is:

> The HSL bus reliability project successfully evaluates clustering and anomaly detection models using suitable metrics. KMeans and DBSCAN reveal reliability groups and hotspot areas, while Isolation Forest and LOF identify unusual operation windows. The results are visualized through tables, plots, CartoDB-style maps, and the dashboard. The analysis can help planners monitor bus reliability, investigate hotspots, improve realtime data quality, and plan future predictive modelling once more GTFS-RT history is collected.
